# import libs

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import os
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# read data

In [48]:
def read_nem_data(data_type, start_date, end_date, base_dir="extracted_csv"):
    """
    Reads NEM data files based on data_type and date range, handling format changes before and after Aug 2024.
    
    Parameters:
        data_type (str): Type of data to load (e.g., "DISPATCHPRICE", "PERDEMAND").
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        base_dir (str): Directory where extracted CSV files are stored.

    Returns:
        pd.DataFrame: Concatenated DataFrame containing the data.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    df_list = []
    
    # Iterate through each month in the date range
    current_date = start_date
    while current_date <= end_date:
        year_month = current_date.strftime("%Y%m")  # Format YYYYMM
        
        # Determine file name format based on date
        if current_date >= pd.to_datetime("2024-08-01"):
            file_path = os.path.join(base_dir, f"PUBLIC_ARCHIVE#{data_type}#FILE01#{year_month}010000.CSV")
        else:
            file_path = os.path.join(base_dir, f"PUBLIC_DVD_{data_type}_{year_month}010000.CSV")
        
        # Read file if it exists
        if os.path.exists(file_path):
            df_temp = pd.read_csv(file_path, skiprows=1)  # Skip first row
            df_list.append(df_temp)
            print(f"✅ Loaded data from {file_path}, rows: {len(df_temp)}")
        else:
            print(f"⚠️ File not found: {file_path}")
        
        # Move to the next month
        current_date += pd.DateOffset(months=1)
    
    # Concatenate all DataFrames
    if df_list:
        df_final = pd.concat(df_list, ignore_index=True)
        print(f"🎉 Successfully concatenated {len(df_final)} rows from all files!")
    else:
        df_final = pd.DataFrame()
        print("❌ No data files found for the given period.")
    
    return df_final

# sample data

In [49]:
def sample_data(df, sample_fraction=0.1, random_state=None):
    """
    Samples a fraction of the raw data without grouping or aggregation.
    
    Parameters:
        df (pd.DataFrame): The raw data.
        sample_fraction (float): The fraction of the data to sample (e.g., 0.1 for 10%, 0.2 for 20%).
        random_state (int or None): The seed for random number generation (optional, for reproducibility).
    
    Returns:
        pd.DataFrame: A sampled subset of the raw data.
    """
    sampled_df = df.sample(frac=sample_fraction, random_state=random_state)
    return sampled_df

# summary table

In [50]:
def summarize_table_info(df):
    summary = {
        "Column": [], "Type": [], "Num Rows": [], "Num Non-Null Rows": [], "Min": [], "Max": [], "Mean": [], "Median": [],
        "Std Dev": [], "Num Unique": [], "Most Freq Value": [], "Most Freq Count": [],
        "Least Freq Value": [], "Least Freq Count": [], "Num Nulls": [], "Num Zeros": []
    }
    
    for col in df.columns:
        summary["Column"].append(col)
        summary["Type"].append(df[col].dtype)
        summary["Num Rows"].append(len(df))
        summary["Num Non-Null Rows"].append(df[col].count())
        
        if df[col].dtype in [np.int64, np.float64]:
            summary["Min"].append(df[col].min())
            summary["Max"].append(df[col].max())
            summary["Mean"].append(df[col].mean())
            summary["Median"].append(df[col].median())
            summary["Std Dev"].append(df[col].std())
            summary["Num Zeros"].append((df[col] == 0).sum())
        else:
            summary["Min"].append(None)
            summary["Max"].append(None)
            summary["Mean"].append(None)
            summary["Median"].append(None)
            summary["Std Dev"].append(None)
            summary["Num Zeros"].append(None)
        
        summary["Num Unique"].append(df[col].nunique())
        most_freq = df[col].mode()
        least_freq = df[col].value_counts().idxmin() if not df[col].isna().all() else None
        
        summary["Most Freq Value"].append(most_freq.iloc[0] if not most_freq.empty else None)
        summary["Most Freq Count"].append(df[col].value_counts().max() if not df[col].isna().all() else None)
        summary["Least Freq Value"].append(least_freq)
        summary["Least Freq Count"].append(df[col].value_counts().min() if not df[col].isna().all() else None)
        summary["Num Nulls"].append(df[col].isna().sum())
    
    summary_df = pd.DataFrame(summary)
    return summary_df.style.background_gradient(cmap="coolwarm", axis=None)

# distribution plot

In [51]:
def distribution_plot_seaborn(df, column, title="Distribution Plot", bins=40, kde=True, color='blue'):
    """
    Creates a distribution plot (histogram with KDE) using Seaborn.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the data.
        column (str): The column name for which the distribution plot is to be created.
        title (str): The title of the plot (default: "Distribution Plot").
        bins (int): The number of bins for the histogram (default: 20).
        kde (bool): Whether to plot the KDE curve (default: True).
        color (str): Color for the plot (default: 'blue').
        
    Returns:
        matplotlib.figure.Figure: The figure containing the distribution plot.
    """
    plt.figure(figsize=(10, 6))

    # Plotting histogram and KDE (if kde=True)
    sns.histplot(df[column], bins=bins, kde=kde, color=color, stat="density", line_kws={'linewidth': 2})

    # Customize the plot with title and axis labels
    plt.title(title, fontsize=16)
    plt.xlabel(column, fontsize=14)
    plt.ylabel('Density', fontsize=14)
    
    # Display the plot
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return plt.gcf()

# plot pie chart 

In [52]:
def plot_pie_chart(df, column_name, value_column):
    """
    Plots a pie chart using Plotly, aggregating values based on a specified column.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The column to group by (e.g., "Fuel Source - Descriptor").
        value_column (str): The column to sum (e.g., "SCADAVALUE").
    """
    # Group by the specified column and sum the value_column (e.g., SCADAVALUE)
    category_sums = df.groupby(column_name)[value_column].sum().reset_index()

    # Plot the pie chart with Plotly
    fig = px.pie(
        category_sums, 
        names=column_name, 
        values=value_column, 
        title=f"Pie Chart of {column_name} by {value_column} Sum with Percentages",
    )

    fig.show()

# plot stacked area chart with time

In [53]:
def plot_stacked_area_plotly(df, time_column, category_column, value_column, time_grain="M"):
    """
    Plots a stacked area chart using Plotly.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        time_column (str): The datetime column.
        category_column (str): The column representing categories.
        value_column (str): The column representing numerical values.
        time_grain (str): The time aggregation level ("D"=day, "W"=week, "M"=month, "Y"=year).
    """
    # Convert time column to datetime if not already
    df[time_column] = pd.to_datetime(df[time_column])

    # Define time formats for grouping
    time_format = {"D": "%Y-%m-%d", "W": "%Y-%W", "M": "%Y-%m", "Y": "%Y"}

    # Remove Feb2025
    df["MonthYear"] = df[time_column].dt.strftime("%Y-%m")
    df = df[df["MonthYear"] != "2025-02"]
    
    # Format time column based on chosen grain
    df["Time_Group"] = df[time_column].dt.strftime(time_format.get(time_grain, "%Y-%m"))

    # Aggregate the data
    df_grouped = df.groupby(["Time_Group", category_column])[value_column].sum().reset_index()

    # Calculate percentage within each time step
    df_grouped["Percentage"] = df_grouped.groupby("Time_Group")[value_column].transform(lambda x: x / x.sum() * 100)

    # Convert percentage to string format for display
    df_grouped["Percentage_Text"] = df_grouped["Percentage"].map(lambda x: f"{x:.1f}%")
    
    # Plot using Plotly Express
    fig = px.area(df_grouped, 
                  x="Time_Group", 
                  y=value_column, 
                  color=category_column, 
                  title=f"Stacked Area Chart of {category_column} ({time_grain})",
                  labels={value_column: "SCADAVALUE", "Time_Group": "Time"},
                  line_group=category_column,
                  text=df_grouped["Percentage_Text"])  # Show percentage as text

    # Update layout for better readability
    fig.update_traces(textposition="middle center")  # Adjust position of percentage labels
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title="SCADAVALUE",
        legend_title=category_column,
        xaxis=dict(tickangle=-45),
        hovermode="x unified",
        width=1000, 
        height=600
    )

    fig.show()

In [54]:
def plot_stacked_area_plotly_sumorsvg(df, time_column, category_column, value_column, time_grain="M", agg_func="sum"):
    """
    Plots a stacked area chart using Plotly with either sum or average aggregation.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        time_column (str): The datetime column.
        category_column (str): The column representing categories.
        value_column (str): The column representing numerical values.
        time_grain (str): The time aggregation level ("D"=day, "W"=week, "M"=month, "Y"=year).
        agg_func (str): Aggregation function ("sum" or "avg").
    """
    # Convert time column to datetime if not already
    df[time_column] = pd.to_datetime(df[time_column])

    # Define time formats for grouping
    time_format = {"D": "%Y-%m-%d", "W": "%Y-%W", "M": "%Y-%m", "Y": "%Y"}

    # Remove Feb2025
    df["MonthYear"] = df[time_column].dt.strftime("%Y-%m")
    df = df[df["MonthYear"] != "2025-02"]
    
    # Format time column based on chosen grain
    df["Time_Group"] = df[time_column].dt.strftime(time_format.get(time_grain, "%Y-%m"))

    # Aggregate the data based on the aggregation function (sum or avg)
    if agg_func == "sum":
        df_grouped = df.groupby(["Time_Group", category_column])[value_column].sum().reset_index()
    elif agg_func == "avg":
        df_grouped = df.groupby(["Time_Group", category_column])[value_column].mean().reset_index()
    else:
        raise ValueError("agg_func must be 'sum' or 'avg'")

    # Calculate percentage within each time step
    df_grouped["Percentage"] = df_grouped.groupby("Time_Group")[value_column].transform(lambda x: x / x.sum() * 100)

    # Convert percentage to string format for display
    df_grouped["Percentage_Text"] = df_grouped["Percentage"].map(lambda x: f"{x:.1f}%")
    
    # Plot using Plotly Express
    fig = px.area(df_grouped, 
                  x="Time_Group", 
                  y=value_column, 
                  color=category_column, 
                  title=f"Stacked Area Chart of {category_column} ({agg_func.capitalize()}) - {time_grain}",
                  labels={value_column: "SCADAVALUE", "Time_Group": "Time"},
                  line_group=category_column,
                  text=df_grouped["Percentage_Text"])  # Show percentage as text

    # Update layout for better readability
    fig.update_traces(textposition="middle center")  # Adjust position of percentage labels
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title="SCADAVALUE",
        legend_title=category_column,
        xaxis=dict(tickangle=-45),
        hovermode="x unified",
        width=1000, 
        height=600
    )

    fig.show()

# plot stacked bar chart with time

In [55]:
def plot_stacked_bar_chart(df, time_column, category_column, value_column, time_grain="M"):
    """
    Plots a stacked bar chart using Plotly with percentage contribution to the total for each time period.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        time_column (str): The datetime column.
        category_column (str): The column representing categories.
        value_column (str): The column representing numerical values.
        time_grain (str): The time aggregation level ("D"=day, "W"=week, "M"=month, "DOW"=day of the week).
    """
    # Convert time column to datetime if not already
    df[time_column] = pd.to_datetime(df[time_column])

    # Define time formats for grouping
    time_format = {"D": "%Y-%m-%d", "W": "%Y-%W", "M": "%m", "Y": "%Y", "DOW": "%A"}

    # Format time column based on chosen grain
    df["Time_Group"] = df[time_column].dt.strftime(time_format.get(time_grain, "%Y-%m"))

    # Aggregate the data (sum the values for each time group and category)
    df_grouped = df.groupby(["Time_Group", category_column])[value_column].sum().reset_index()

    # Calculate the total value for each time group
    df_grouped["Total_Value"] = df_grouped.groupby("Time_Group")[value_column].transform("sum")

    # Calculate the percentage contribution of each category within each time group
    df_grouped["Percentage"] = df_grouped[value_column] / df_grouped["Total_Value"] * 100

    # Format the percentage text to show on the bars
    df_grouped["Percentage_Text"] = df_grouped["Percentage"].map(lambda x: f"{x:.1f}%")

    # Plot using Plotly Express as a stacked bar chart with percentage values
    fig = px.bar(df_grouped, 
                 x="Time_Group", 
                 y=value_column, 
                 color=category_column, 
                 title=f"Stacked Bar Chart of {category_column} ({time_grain})",
                 labels={value_column: "SCADAVALUE", "Time_Group": "Time"},
                 text="Percentage_Text",  # Display numerical value on bars
                 category_orders={"Time_Group": sorted(df["Time_Group"].unique())},  # Sort by Time_Group
                 hover_data={category_column: True, value_column: True})  # Show category and value on hover

    # Update layout for better readability
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title="SCADAVALUE",
        legend_title=category_column,
        xaxis=dict(tickangle=-45),
        hovermode="x unified",
        width=1000, 
        height=600
    )

    # Show the plot
    fig.show()

# plot normal bar chart 

In [56]:
def plot_bar_chart(df, groupby_column, sum_column, title="Bar Chart", xlabel="Group", ylabel="Value", agg_method="sum"):
    """
    Plots a bar chart using Plotly by grouping by one column and either summing or averaging another column.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        groupby_column (str): The column to group by.
        sum_column (str): The column whose values will be aggregated (sum or avg) for each group.
        title (str): The title of the plot.
        xlabel (str): The label for the x-axis.
        ylabel (str): The label for the y-axis.
        agg_method (str): The aggregation method ("sum" or "avg"). Default is "sum".
    """
    # Group by the specified column and apply the aggregation method
    if agg_method == "sum":
        df_grouped = df.groupby(groupby_column)[sum_column].sum().reset_index()
    elif agg_method == "avg":
        df_grouped = df.groupby(groupby_column)[sum_column].mean().reset_index()
    else:
        raise ValueError("Invalid aggregation method. Use 'sum' or 'avg'.")

    # Plot the bar chart using Plotly Express
    fig = px.bar(df_grouped, 
                 x=groupby_column, 
                 y=sum_column, 
                 title=title, 
                 labels={groupby_column: xlabel, sum_column: ylabel})
    
    # Show the plot
    fig.show()

# plot stacked bar chart without time

In [57]:
def plot_stacked_bar_chart_with_percentage(df, groupby_column, category_column, sum_column, title="Stacked Bar Chart with Percentages", xlabel="Group", ylabel="Sum"):
    """
    Plots a stacked bar chart where each segment of the bar represents a category with percentages on the bars.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        groupby_column (str): The column to group by (for the x-axis).
        category_column (str): The column representing categories (for the stacked segments).
        sum_column (str): The column whose values will be summed for each group.
        title (str): The title of the plot.
        xlabel (str): The label for the x-axis.
        ylabel (str): The label for the y-axis.
    """
    # Group by the specified columns and sum the values
    df_grouped = df.groupby([groupby_column, category_column])[sum_column].sum().reset_index()

    # Calculate the total sum for each group to compute percentage
    df_grouped['Total'] = df_grouped.groupby(groupby_column)[sum_column].transform('sum')

    # Calculate the percentage of each category within each group
    df_grouped['Percentage'] = (df_grouped[sum_column] / df_grouped['Total']) * 100

    # Plot the stacked bar chart using Plotly Express
    fig = px.bar(df_grouped, 
                 x=groupby_column, 
                 y=sum_column, 
                 color=category_column,
                 title=title, 
                 labels={groupby_column: xlabel, sum_column: ylabel},
                 text=df_grouped['Percentage'].round(1).astype(str) + '%',  # Show percentage on bars
                 color_discrete_sequence=px.colors.qualitative.Set2)  # Set color scheme
    
    # Adjust layout and display the plot
    fig.update_traces(textposition='inside', texttemplate='%{text}')
    fig.update_layout(barmode='stack', xaxis_title=xlabel, yaxis_title=ylabel, showlegend=True)
    
    # Show the plot
    fig.show()

# plot normal line chart with time and categories

In [58]:
def plot_line_chart_with_categories(df, time_column, category_column, value_column, time_grain="M", agg_method="sum"):
    """
    Plots a line chart with time aggregation and grouping by categories using Plotly.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        time_column (str): The datetime column.
        category_column (str): The column representing categories.
        value_column (str): The column representing numerical values.
        time_grain (str): The time aggregation level ("D"=day, "W"=week, "M"=month, "Y"=year).
        agg_method (str): The aggregation method ("sum" or "avg"). Default is "sum".
    """
    # Convert time column to datetime if not already
    df[time_column] = pd.to_datetime(df[time_column])

    # Define time formats for grouping
    time_format = {"D": "%Y-%m-%d", "W": "%Y-%W", "M": "%Y-%m", "Y": "%Y"}

    # Remove February 2025
    df["MonthYear"] = df[time_column].dt.strftime("%Y-%m")
    df = df[df["MonthYear"] != "2025-02"]

    # Format time column based on chosen grain
    df["Time_Group"] = df[time_column].dt.strftime(time_format.get(time_grain, "%Y-%m"))

    # Aggregate the data based on the chosen method
    if agg_method == "sum":
        df_grouped = df.groupby(["Time_Group", category_column])[value_column].sum().reset_index()
    elif agg_method == "avg":
        df_grouped = df.groupby(["Time_Group", category_column])[value_column].mean().reset_index()
    else:
        raise ValueError("Invalid aggregation method. Use 'sum' or 'avg'.")

    # Convert Time_Group to datetime for proper sorting
    df_grouped["Time_Group"] = pd.to_datetime(df_grouped["Time_Group"])

    # Plot using Plotly Express
    fig = px.line(df_grouped, x="Time_Group", y=value_column, color=category_column, 
                  title=f"Line Chart of {category_column} ({time_grain}, {agg_method})",
                  labels={"Time_Group": "Time", value_column: "SCADAVALUE"},
                  markers=True)

    # Update layout for better readability
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title="SCADAVALUE",
        legend_title=category_column,
        xaxis=dict(tickangle=-45),
        width=1000, 
        height=600
    )

    # Show the plot
    fig.show()

# plot normal line chart with time and no categories

In [59]:
def plot_line_chart_time(df, time_column, value_column, time_grain="M", agg_method="sum"):
    """
    Plots a line chart by grouping data only by the time column using Plotly.

    Parameters:
        df (pd.DataFrame): The dataframe containing the data.
        time_column (str): The datetime column.
        value_column (str): The column representing numerical values.
        time_grain (str): The time aggregation level ("D"=day, "W"=week, "M"=month, "Y"=year).
        agg_method (str): The aggregation method ("sum" or "avg"). Default is "sum".
    """
    # Convert time column to datetime
    df[time_column] = pd.to_datetime(df[time_column])

    # Define time formats for grouping
    time_format = {"D": "%Y-%m-%d", "W": "%Y-%W", "M": "%Y-%m", "Y": "%Y"}

    # Remove February 2025
    df["MonthYear"] = df[time_column].dt.strftime("%Y-%m")
    df = df[df["MonthYear"] != "2025-02"]
    
    # Format time column based on chosen grain
    df["Time_Group"] = df[time_column].dt.strftime(time_format.get(time_grain, "%Y-%m"))

    # Aggregate the data based on the chosen method
    if agg_method == "sum":
        df_grouped = df.groupby("Time_Group")[value_column].sum().reset_index()
    elif agg_method == "avg":
        df_grouped = df.groupby("Time_Group")[value_column].mean().reset_index()
    else:
        raise ValueError("Invalid aggregation method. Use 'sum' or 'avg'.")

    # Convert Time_Group to datetime for correct plotting order
    df_grouped["Time_Group"] = pd.to_datetime(df_grouped["Time_Group"])

    # Plot using Plotly Express
    fig = px.line(df_grouped, 
                  x="Time_Group", 
                  y=value_column, 
                  title=f"Line Chart ({time_grain}, {agg_method})",
                  labels={"Time_Group": "Time", value_column: value_column},
                  markers=True)

    # Update layout for better readability
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title=value_column,
        xaxis=dict(tickangle=45),
        template="plotly",
        height=600
    )

    # Show the plot
    fig.show()

# plot boxplot with 1 catecol and 1 valuecol

In [60]:
def plot_boxplot_by_category_plt(df, groupby_column, value_column, title="Boxplot", xlabel="Group", ylabel="Value", agg_func="mean"):
    """
    Function to plot a boxplot grouped by a categorical column with respect to a value column using Matplotlib and Seaborn.

    Parameters:
        df (DataFrame): The input pandas DataFrame.
        groupby_column (str): The column to group by (for the x-axis).
        value_column (str): The column representing numerical values.
        title (str): The title of the plot.
        xlabel (str): The label for the x-axis.
        ylabel (str): The label for the y-axis.
        agg_func (str): Aggregation function ('sum', 'avg', or 'count') for grouping.
    """
    # Create a new column 'Holiday_Status' based on 'Holidays' column
    df['Holiday_Status'] = df['Holidays'].apply(lambda x: 'Holiday' if pd.notna(x) else 'Non-holiday')
    
    # Aggregate the data based on 'agg_func'
    if agg_func == 'mean':
        df_agg = df.groupby([groupby_column])[value_column].mean().reset_index()
    elif agg_func == 'sum':
        df_agg = df.groupby([groupby_column])[value_column].sum().reset_index()
    elif agg_func == 'count':
        df_agg = df.groupby([groupby_column])[value_column].count().reset_index()
    else:
        raise ValueError("agg_func must be 'sum', 'avg', or 'count'")

    # Plotting using Seaborn (Boxplot)
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=groupby_column, y=value_column, data=df, palette="Set2")

    # Title and labels
    plt.title(title, fontsize=16)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)

    # Show plot
    plt.show()